# H3 Studio · Colab
选择 L4 GPU 和高 RAM，再运行下方单元格。输入 Gmail 邮箱，按提示配置应用专用密码。权重约 29.4 GB，保存在 Colab。
Select L4 GPU and High RAM, then run the cell below. Enter your Gmail address and follow the app-password instructions. About 29.4 GB of weights will be downloaded to Colab.
[中文说明 / English guide](https://github.com/bennix/H3Studio)


In [ ]:
import shutil
import subprocess
nvidia_smi = shutil.which('nvidia-smi')
if not nvidia_smi:
    raise RuntimeError('当前运行时未提供 NVIDIA GPU。请在 Colab 菜单“代码执行程序 → 更改运行时类型”选择 L4 GPU（建议开启高 RAM）并保存，连接后重新运行本单元格。')
try:
    gpu_details = subprocess.check_output([nvidia_smi, '--query-gpu=name,memory.total,memory.free', '--format=csv'], text=True, stderr=subprocess.STDOUT)
except subprocess.CalledProcessError as error:
    raise RuntimeError('GPU 驱动尚未就绪，请重新连接 GPU 运行时后再运行。本次未开始模型下载。') from error
print(gpu_details, flush=True)
from google.colab import userdata
import re
H3_NOTIFICATION_EMAIL = input('接收登录信息的 Gmail 邮箱（也是发件账号）：').strip()
if not re.fullmatch('[^\\s@]+@gmail\\.com', H3_NOTIFICATION_EMAIL, re.IGNORECASE):
    raise ValueError('请输入 Gmail 邮箱')

def get_h3_gmail_password():
    cached = globals().get('H3_GMAIL_APP_PASSWORD', '')
    if cached:
        return cached
    try:
        from google.colab import userdata
        value = userdata.get('COMFY_GMAIL_APP_PASSWORD')
    except Exception as error:
        reason = type(error).__name__
        if reason == 'NotebookAccessError':
            print('已有邮件密钥尚未授权本笔记本，可在左侧 Secret 开启访问权限，或在下方临时填写。')
        elif reason == 'SecretNotFoundError':
            print('当前 Colab 未配置 Gmail 应用专用密码，请通过下方链接生成后填写。')
        else:
            print('暂时无法读取 Colab 邮件密钥，可在下方临时填写 Gmail 应用专用密码。')
        from IPython.display import display, HTML
        display(HTML('<section><h3>生成 Gmail 应用专用密码</h3><ol><li>登录上方填写的 Gmail 账号，打开 <a href="https://myaccount.google.com/signinoptions/two-step-verification" target="_blank" rel="noopener">Google 两步验证</a>，按页面提示完成开启。</li><li>打开 <a href="https://myaccount.google.com/apppasswords" target="_blank" rel="noopener">应用专用密码</a>。Google 可能要求重新登录。</li><li>在“应用名称”填写 H3 Studio，点击“创建”。</li><li>复制 Google 显示的 16 位应用专用密码，粘贴到下方隐藏输入框并回车。密码中的空格会自动去除。</li></ol><p>这里填写 Google 生成的应用专用密码。临时输入仅用于当前 Colab 运行时，重建运行时后需再次填写。也可将其保存为 Colab Secret COMFY_GMAIL_APP_PASSWORD，并开启本笔记本访问权限。</p></section>'))
        import getpass
        value = getpass.getpass('Gmail 应用专用密码（隐藏输入，仅保存在当前运行时内存）：')
    value = value.replace(' ', '').strip()
    if not value:
        raise RuntimeError('未填写 Gmail 应用专用密码。请配置 Secret 或在隐藏输入框填写后重试。')
    globals()['H3_GMAIL_APP_PASSWORD'] = value
    return value
H3_GMAIL_APP_PASSWORD = get_h3_gmail_password()
import requests
h3_is_running = False
try:
    response = requests.get('http://127.0.0.1:8188/object_info/H3ReferenceEdit', timeout=3)
    results_api = requests.get('http://127.0.0.1:8188/h3-studio/results', timeout=3)
    h3_is_running = response.ok and 'H3ReferenceEdit' in response.json() and results_api.ok
except requests.RequestException:
    pass
if h3_is_running:
    print('H3 参考编辑服务已运行，正在验证公网入口。', flush=True)
else:
    exec("# 13. MiniMax H3 Image：安装并下载至 Colab 远端（W4A8 + NVFP4）\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport urllib.request\n\nROOT = Path('/content/ComfyUI')\n# 空白 Colab 默认是 CPU，必须在任何邮箱输入或下载之前检查 GPU。\nimport shutil\nimport subprocess\nnvidia_smi = shutil.which('nvidia-smi')\nif not nvidia_smi:\n    raise RuntimeError('当前运行时未提供 NVIDIA GPU。请在 Colab 菜单“代码执行程序 → 更改运行时类型”选择 L4 GPU（建议开启高 RAM）并保存，连接后重新运行本单元格。')\ntry:\n    gpu_details = subprocess.check_output([nvidia_smi, '--query-gpu=name,memory.total,memory.free', '--format=csv'], text=True, stderr=subprocess.STDOUT)\nexcept subprocess.CalledProcessError as error:\n    raise RuntimeError('GPU 驱动尚未就绪，请重新连接 GPU 运行时后再运行。本次未开始模型下载。') from error\nprint(gpu_details, flush=True)\n\nif not ROOT.exists():\n    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(ROOT)], check=True)\nelse:\n    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)\nwith open('/content/minimax-h3-image-install.log', 'w') as log:\n    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], stdout=log, stderr=subprocess.STDOUT)\nif result.returncode:\n    print(Path('/content/minimax-h3-image-install.log').read_text()[-8000:])\n    raise RuntimeError('ComfyUI 依赖安装失败')\nprint('ComfyUI 已更新，开始下载 H3 Image 模型。', flush=True)\n\nrepo = 'Comfy-Org/MiniMax-H3-Image'\nfiles = [\n    'diffusion_models/minimax_h3_image_pruned_w4a8.safetensors',\n    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',\n    'vae/minimax_h3_image_vae_fp16.safetensors',\n]\nwith urllib.request.urlopen(f'https://huggingface.co/api/models/{repo}/tree/main?recursive=true&limit=1000') as response:\n    metadata = {item['path']: item for item in json.load(response)}\nfor relative in files:\n    info = metadata[relative]\n    target = ROOT / 'models' / relative\n    target.parent.mkdir(parents=True, exist_ok=True)\n    expected = info['size']\n    expected_sha = info.get('lfs', {}).get('oid')\n    if not target.exists() or target.stat().st_size != expected:\n        partial = target.with_suffix(target.suffix + '.part')\n        url = f'https://huggingface.co/{repo}/resolve/main/{relative}'\n        print(f'下载到 Colab：{target.name}，{expected / 1e9:.2f} GB', flush=True)\n        for attempt in range(4):\n            offset = partial.stat().st_size if partial.exists() else 0\n            try:\n                request = urllib.request.Request(url, headers={'Range': f'bytes={offset}-'} if offset else {})\n                with urllib.request.urlopen(request, timeout=120) as response:\n                    resume = offset and response.status == 206\n                    received = offset if resume else 0\n                    report_at = received + 2 * 1024**3\n                    with partial.open('ab' if resume else 'wb') as output:\n                        while chunk := response.read(8 * 1024**2):\n                            output.write(chunk)\n                            received += len(chunk)\n                            if received >= report_at:\n                                print(f'  {target.name} {received / expected:.0%}', flush=True)\n                                report_at += 2 * 1024**3\n                if partial.stat().st_size != expected:\n                    raise RuntimeError('下载字节数不匹配')\n                partial.replace(target)\n                break\n            except Exception as error:\n                if attempt == 3:\n                    raise\n                print(f'下载重试 {attempt + 1}：{type(error).__name__}', flush=True)\n                time.sleep(3)\n    if expected_sha:\n        digest = hashlib.file_digest(target.open('rb'), 'sha256').hexdigest()\n        if digest != expected_sha:\n            raise RuntimeError(f'{target.name} SHA256 校验失败')\n    print(f'已验证 {target.name}，{target.stat().st_size:,} 字节', flush=True)\nprint('H3 Image 三项权重已全部保存到 /content/ComfyUI/models，总计约 29.4 GB。', flush=True)\n", globals())
    exec('# 14. MiniMax H3 Image：启动 ComfyUI 并保存可视化生图工作流\nfrom pathlib import Path\nimport json\nimport subprocess\nimport sys\nimport time\nimport requests\nROOT = Path(\'/content/ComfyUI\')\np = Path(\'/content/ComfyUI/custom_nodes/full_canvas_export/__init__.py\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'WEB_DIRECTORY = "./web"\\nNODE_CLASS_MAPPINGS = {}\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/full_canvas_export/web/image_compare_classic.js\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'import { app } from "../../scripts/app.js";\\nimport { api } from "../../scripts/api.js";\\n\\n// ImageCompare ships with a Vue-only display. Paint the same saved images on\\n// the classic canvas so the comparison also appears in full-canvas PNG exports.\\nconst outputs = new Map();\\nconst images = new Map();\\nfunction url(meta) {\\n  return api.apiURL(\\\'/view?\\\' + new URLSearchParams(meta));\\n}\\nfunction image(src) {\\n  if (!src) return null;\\n  if (!images.has(src)) {\\n    const img = new Image();\\n    img.onload = () => app.graph?.setDirtyCanvas(true, true);\\n    img.src = src;\\n    images.set(src, img);\\n  }\\n  const img = images.get(src);\\n  return img.complete && img.naturalWidth ? img : null;\\n}\\nfunction source(node, side) {\\n  const saved = outputs.get(String(node.id)) ?? app.nodeOutputs?.[node.id];\\n  const meta = saved?.[side + \\\'_images\\\']?.[0];\\n  if (meta) return url(meta);\\n  const slot = node.inputs?.findIndex(input => input.name === \\\'image_\\\' + side);\\n  const upstream = slot >= 0 ? node.getInputNode(slot) : null;\\n  if (!upstream) return null;\\n  const out = outputs.get(String(upstream.id)) ?? app.nodeOutputs?.[upstream.id];\\n  if (out?.images?.[0]) return url(out.images[0]);\\n  if (upstream.imgs?.[0]?.src) return upstream.imgs[0].src;\\n  const file = upstream.widgets?.find(w => w.name === \\\'image\\\')?.value;\\n  return typeof file === \\\'string\\\' ? url({filename:file, type:\\\'input\\\'}) : null;\\n}\\nfunction install(node) {\\n  if ((node.comfyClass ?? node.constructor.comfyClass ?? node.type) !== \\\'ImageCompare\\\') return;\\n  const widget = node.widgets?.find(w => w.type === \\\'imagecompare\\\');\\n  if (!widget || widget._classicCompare) return;\\n  widget._classicCompare = true;\\n  let split = 0.5;\\n  widget.draw = function(ctx, node, width, y) {\\n    const x = 15, w = width - 30, h = Math.max(100, node.size[1] - y - 15);\\n    const a = image(source(node, \\\'a\\\')), b = image(source(node, \\\'b\\\'));\\n    ctx.save();\\n    ctx.fillStyle = \\\'#181818\\\'; ctx.fillRect(x,y,w,h);\\n    ctx.beginPath(); ctx.rect(x,y,w,h); ctx.clip();\\n    function paint(img) {\\n      if (!img) return;\\n      const scale = Math.min(w/img.naturalWidth, h/img.naturalHeight);\\n      const iw = img.naturalWidth*scale, ih = img.naturalHeight*scale;\\n      ctx.drawImage(img,x+(w-iw)/2,y+(h-ih)/2,iw,ih);\\n    }\\n    paint(b ?? a);\\n    if (a && b) {\\n      ctx.save(); ctx.beginPath(); ctx.rect(x,y,w*split,h); ctx.clip(); paint(a); ctx.restore();\\n      ctx.strokeStyle = \\\'#fff\\\'; ctx.lineWidth = 2;\\n      ctx.beginPath(); ctx.moveTo(x+w*split,y); ctx.lineTo(x+w*split,y+h); ctx.stroke();\\n    }\\n    ctx.fillStyle = \\\'#ffffff\\\'; ctx.font = \\\'12px sans-serif\\\'; ctx.textAlign=\\\'left\\\';\\n    ctx.fillText(a || b ? \\\'A ← 移动鼠标对比 → B\\\' : \\\'等待输入图像或执行结果\\\', x+8,y+18);\\n    ctx.restore();\\n  };\\n  const oldDown = node.onMouseDown, oldMove = node.onMouseMove;\\n  function update(pos) {\\n    const y = widget.last_y ?? widget.y ?? 0;\\n    if (pos[0] < 15 || pos[0] > node.size[0]-15 || pos[1] < y || pos[1] > node.size[1]-15) return false;\\n    split = Math.max(0,Math.min(1,(pos[0]-15)/(node.size[0]-30)));\\n    node.setDirtyCanvas(true,true);\\n    return true;\\n  }\\n  node.onMouseDown = function(event,pos,...rest) {\\n    return update(pos) || oldDown?.call(this,event,pos,...rest);\\n  };\\n  node.onMouseMove = function(event,pos,...rest) {\\n    update(pos);\\n    return oldMove?.call(this,event,pos,...rest);\\n  };\\n}\\napp.registerExtension({\\n  name: \\\'Local.ClassicImageCompare\\\',\\n  nodeCreated: install,\\n  async setup() {\\n    api.addEventListener(\\\'executed\\\', ({detail}) => {\\n      outputs.set(String(detail.display_node ?? detail.node), detail.output);\\n      app.graph?.setDirtyCanvas(true,true);\\n    });\\n    // Restore completed results after refresh without another GPU generation.\\n    try {\\n      const response = await api.fetchApi(\\\'/history?max_items=10\\\');\\n      if (!response.ok) return;\\n      const history = await response.json();\\n      for (const item of Object.values(history).sort((a,b)=>(a.prompt?.[0]??0)-(b.prompt?.[0]??0))) {\\n        for (const [id,out] of Object.entries(item.outputs ?? {})) outputs.set(id,out);\\n      }\\n      app.graph?.setDirtyCanvas(true,true);\\n    } catch (error) { console.warn(\\\'Classic comparison history:\\\',error); }\\n  },\\n});\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/full_canvas_export/web/full_canvas_export.js\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'import { app } from "../../scripts/app.js";\\n\\n// Render at one output pixel per graph coordinate, independently of screen zoom.\\nasync function renderFullCanvas() {\\n  const view = app.canvas;\\n  const graph = view.graph;\\n  const nodes = graph?._nodes ?? [];\\n  if (!nodes.length) throw new Error("当前画布没有节点。");\\n  view.draw(true, true);\\n  await Promise.resolve();\\n  const bounds = nodes.map(node => Array.from(node.getBounding(undefined, true)));\\n  for (const group of graph._groups ?? []) bounds.push(Array.from(group._bounding));\\n  for (const reroute of graph.reroutes?.values?.() ?? []) {\\n    const pos = reroute.pos ?? reroute._pos;\\n    if (pos) bounds.push([pos[0] - 10, pos[1] - 10, 20, 20]);\\n  }\\n  const minX = Math.min(...bounds.map(b => b[0]));\\n  const minY = Math.min(...bounds.map(b => b[1]));\\n  const maxX = Math.max(...bounds.map(b => b[0] + b[2]));\\n  const maxY = Math.max(...bounds.map(b => b[1] + b[3]));\\n  // LiteGraph spline handles can extend by a quarter of the link length.\\n  const padding = Math.max(80, Math.ceil(Math.hypot(maxX - minX, maxY - minY) / 4) + 24);\\n  const left = Math.floor(minX - padding);\\n  const top = Math.floor(minY - padding);\\n  const width = Math.ceil(maxX + padding - left);\\n  const height = Math.ceil(maxY + padding - top);\\n  const dpr = window.devicePixelRatio || 1;\\n  if (Math.max(width, height) * dpr > 16384 || width * height * dpr * dpr > 64000000) {\\n    throw new Error(`画布过大（${width}×${height}），请缩小节点间距后再导出；不会自动缩小图片。`);\\n  }\\n  const canvas = view.canvas;\\n  const ctx = view.ctx;\\n  const state = {\\n    width: canvas.width, height: canvas.height,\\n    scale: view.ds.scale, offset: Array.from(view.ds.offset),\\n    transform: ctx.getTransform(), viewport: view.viewport,\\n    viewportSize: view.ds.getViewportSize?.(), dirtyArea: view.dirty_area,\\n    showInfo: view.show_info,\\n  };\\n  const patches = [];\\n  try {\\n    // DOM textareas are not part of Canvas.toBlob, so paint them during this draw.\\n    for (const node of nodes) for (const widget of node.widgets ?? []) {\\n      if (widget.type !== "customtext" || typeof widget.draw !== "function") continue;\\n      const original = widget.draw;\\n      const own = Object.prototype.hasOwnProperty.call(widget, "draw");\\n      widget.draw = function(context, node, widgetWidth, y, widgetHeight, ...rest) {\\n        original.call(this, context, node, widgetWidth, y, widgetHeight, ...rest);\\n        const element = this.element ?? this.inputEl;\\n        const text = String(element?.value ?? this.value ?? "");\\n        const margin = this.margin ?? 10;\\n        const w = widgetWidth - margin * 2;\\n        const h = (this.computedHeight ?? widgetHeight ?? 80) - margin * 2;\\n        const style = element ? getComputedStyle(element) : null;\\n        context.save();\\n        context.beginPath();\\n        context.rect(margin, y + margin, w, h);\\n        context.clip();\\n        context.fillStyle = style?.backgroundColor || "#222";\\n        context.fillRect(margin, y + margin, w, h);\\n        context.fillStyle = style?.color || "#eee";\\n        context.font = style?.font || "12px monospace";\\n        context.textBaseline = "top";\\n        const lineHeight = parseFloat(style?.lineHeight) || (parseFloat(style?.fontSize) || 12) * 1.35;\\n        let textY = y + margin + 4;\\n        for (const paragraph of text.split("\\\\n")) {\\n          let line = "";\\n          for (const character of paragraph) {\\n            if (line && context.measureText(line + character).width > w - 8) {\\n              context.fillText(line, margin + 4, textY);\\n              textY += lineHeight;\\n              line = "";\\n            }\\n            line += character;\\n          }\\n          context.fillText(line, margin + 4, textY);\\n          textY += lineHeight;\\n          if (textY > y + margin + h) break;\\n        }\\n        context.restore();\\n      };\\n      patches.push(() => { if (own) widget.draw = original; else delete widget.draw; });\\n    }\\n    canvas.width = Math.ceil(width * dpr);\\n    canvas.height = Math.ceil(height * dpr);\\n    ctx.setTransform(dpr, 0, 0, dpr, 0, 0);\\n    view.viewport = undefined;\\n    view.dirty_area = undefined;\\n    view.ds.setViewportSize?.(width, height);\\n    view.show_info = false;\\n    view.ds.scale = 1;\\n    view.ds.offset = [-left, -top];\\n    view.draw(true, true);\\n    if (view.visible_nodes.length !== nodes.length) {\\n      throw new Error(`渲染范围不完整：${view.visible_nodes.length}/${nodes.length} 个节点，已取消导出。`);\\n    }\\n    // ComfyUI queues preview-image drawing in microtasks; flush before snapshot.\\n    await Promise.resolve();\\n    // Restore before the next animation frame or awaiting PNG encoding.\\n    const output = document.createElement("canvas");\\n    output.width = width;\\n    output.height = height;\\n    const out = output.getContext("2d");\\n    out.fillStyle = view.clear_background_color || "#181818";\\n    out.fillRect(0, 0, width, height);\\n    out.drawImage(canvas, 0, 0, width, height);\\n    return output;\\n  } finally {\\n    for (const undo of patches) undo();\\n    canvas.width = state.width;\\n    canvas.height = state.height;\\n    ctx.setTransform(state.transform);\\n    view.viewport = state.viewport;\\n    view.dirty_area = state.dirtyArea;\\n    if (state.viewportSize) view.ds.setViewportSize?.(...state.viewportSize);\\n    view.show_info = state.showInfo;\\n    view.ds.scale = state.scale;\\n    view.ds.offset = state.offset;\\n    view.draw(true, true);\\n  }\\n}\\n\\napp.registerExtension({\\n  name: "Local.FullCanvasPNG",\\n  setup() {\\n    const button = document.createElement("button");\\n    button.id = "full-canvas-export";\\n    button.textContent = "全画布 PNG · 100%";\\n    button.title = "以 100% 比例导出当前画布全部节点、连线和文本，不受屏幕范围限制";\\n    button.style.cssText = "position:fixed;left:76px;bottom:18px;z-index:1000;padding:9px 14px;border:1px solid #666;border-radius:7px;background:#252525;color:#fff;font:13px sans-serif;cursor:pointer;box-shadow:0 2px 8px #0006";\\n    button.onclick = async () => {\\n      button.disabled = true;\\n      button.textContent = "正在导出…";\\n      try {\\n        await document.fonts.ready;\\n        const image = await renderFullCanvas();\\n        const blob = await new Promise(resolve => image.toBlob(resolve, "image/png"));\\n        if (!blob) throw new Error("浏览器无法编码此画布。");\\n        const url = URL.createObjectURL(blob);\\n        const link = document.createElement("a");\\n        link.href = url;\\n        link.download = `ComfyUI-full-canvas-100pct-${image.width}x${image.height}.png`;\\n        link.click();\\n        setTimeout(() => URL.revokeObjectURL(url), 60000);\\n        button.textContent = `已导出 ${image.width}×${image.height}`;\\n      } catch (error) {\\n        console.error("Full canvas export:", error);\\n        window.alert("全画布导出失败：" + error.message);\\n      } finally {\\n        button.disabled = false;\\n        setTimeout(() => { button.textContent = "全画布 PNG · 100%"; }, 4000);\\n      }\\n    };\\n    document.body.appendChild(button);\\n  },\\n});\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/full_canvas_export/web/compute_units.js\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'import { app } from "../../scripts/app.js";\\n\\nconst endpoint = "/userdata/colab_compute_units.json";\\nconst notebook = "https://colab.research.google.com/";\\n\\nfunction valid(data) {\\n  return data && Number.isFinite(data.balance) && data.balance >= 0 &&\\n    Number.isFinite(data.rate_per_hour) && data.rate_per_hour >= 0 &&\\n    Number.isFinite(data.observed_at_ms) && data.observed_at_ms > 0;\\n}\\n\\napp.registerExtension({\\n  name: "Local.ColabComputeUnits",\\n  setup() {\\n    let baseline;\\n    const panel = document.createElement("section");\\n    panel.id = "colab-compute-units";\\n    panel.setAttribute("aria-label", "Colab 剩余计算单元");\\n    panel.title = "依据最近核对的余额和当前运行时费率估算；其他会话、充值、额度到期和费率变化不计入。实际余额以 Colab 为准。";\\n    panel.style.cssText = "position:fixed;right:18px;bottom:78px;z-index:1000;width:260px;padding:12px;border:1px solid #666;border-radius:8px;background:#202020ed;color:#eee;font:12px/1.6 sans-serif;box-shadow:0 3px 12px #0007";\\n    panel.innerHTML = `<button type="button" data-action="toggle" style="position:absolute;right:9px;top:9px;background:none;border:0;color:#c8dfff;cursor:pointer;font:12px sans-serif"></button>\\n      <div data-field="heading" style="color:#bbb">剩余计算单元（估算）</div>\\n      <strong data-field="remaining" style="font-size:23px;font-variant-numeric:tabular-nums">读取中…</strong>\\n      <div data-field="details">\\n      <div data-field="base" style="color:#bbb"></div>\\n      <div data-field="checked" style="color:#aaa"></div>\\n      <div style="color:#aaa">仅按当前运行时估算 · 官方余额为准</div>\\n      <a href="${notebook}" target="_blank" rel="noopener" style="color:#9bc6ff">查看 Colab 官方余额</a>\\n      <details style="margin-top:5px"><summary style="cursor:pointer">校准余额 / 费率</summary>\\n        <form style="display:grid;gap:5px;margin-top:6px">\\n          <label>Colab 当前余额 <input name="balance" aria-label="Colab 当前余额" type="number" min="0" step="0.01" required style="width:100%;color:#eee;background:#333;border:1px solid #666;border-radius:3px"></label>\\n          <label>每小时消耗 <input name="rate" aria-label="每小时消耗计算单元" type="number" min="0" step="0.01" required style="width:100%;color:#eee;background:#333;border:1px solid #666;border-radius:3px"></label>\\n          <button type="submit" style="background:#3c6297;color:white;border:0;border-radius:4px;padding:5px;cursor:pointer">保存校准值</button>\\n        </form>\\n      </details><div data-field="message" role="status" style="color:#c8dfff"></div></div>`;\\n    document.body.appendChild(panel);\\n    const field = name => panel.querySelector(`[data-field="${name}"]`);\\n    const form = panel.querySelector("form");\\n    const toggle = panel.querySelector(\\\'[data-action="toggle"]\\\');\\n    let collapsed = localStorage.getItem("colab-compute-units-collapsed") !== "false";\\n    function applyCollapsed() {\\n      field("heading").hidden = collapsed;\\n      field("details").hidden = collapsed;\\n      field("remaining").style.fontSize = collapsed ? "16px" : "23px";\\n      panel.style.width = collapsed ? "auto" : "260px";\\n      panel.style.paddingRight = collapsed ? "52px" : "12px";\\n      toggle.textContent = collapsed ? "展开" : "收起";\\n      toggle.setAttribute("aria-label", collapsed ? "展开余额详情" : "收起余额详情");\\n      toggle.setAttribute("aria-expanded", String(!collapsed));\\n    }\\n    toggle.onclick = () => {\\n      collapsed = !collapsed;\\n      localStorage.setItem("colab-compute-units-collapsed", String(collapsed));\\n      applyCollapsed();\\n    };\\n    applyCollapsed();\\n    function render() {\\n      if (!baseline) { field("remaining").textContent = "待校准"; return; }\\n      const hours = Math.max(0, Date.now() - baseline.observed_at_ms) / 3600000;\\n      const remaining = Math.max(0, baseline.balance - hours * baseline.rate_per_hour);\\n      field("remaining").textContent = `≈${remaining.toFixed(2)} 单元`;\\n      field("base").textContent = `核对基准 ${baseline.balance.toFixed(2)} · ${baseline.rate_per_hour.toFixed(2)} 单元/小时`;\\n      const zone = Intl.DateTimeFormat().resolvedOptions().timeZone;\\n      const zoneLabel = zone === "Asia/Tokyo" ? "日本时间 UTC+9" : zone;\\n      const checked = new Date(baseline.observed_at_ms).toLocaleString("zh-CN", { month: "2-digit", day: "2-digit", hour: "2-digit", minute: "2-digit", hour12: false });\\n      field("checked").textContent = `最近核对 ${checked}（${zoneLabel}）`;\\n      if (hours > 24) field("message").textContent = "基准已超过一天，请核对并校准余额。";\\n    }\\n    async function refresh() {\\n      try {\\n        const response = await fetch(endpoint + "?t=" + Date.now(), { cache: "no-store", credentials: "same-origin" });\\n        if (response.status === 404) { render(); return; }\\n        if (!response.ok) throw new Error("余额基准读取失败");\\n        const data = await response.json();\\n        if (!valid(data)) throw new Error("余额基准格式无效，请校准");\\n        baseline = data;\\n        if (!panel.querySelector("details").open) {\\n          form.elements.balance.value = data.balance;\\n          form.elements.rate.value = data.rate_per_hour;\\n        }\\n        render();\\n      } catch (error) { field("message").textContent = error.message; render(); }\\n    }\\n    form.onsubmit = async event => {\\n      event.preventDefault();\\n      const data = { balance: Number(form.elements.balance.value), rate_per_hour: Number(form.elements.rate.value), observed_at_ms: Date.now(), source: "用户依据 Colab 显示值校准" };\\n      if (!valid(data)) return;\\n      field("message").textContent = "正在保存…";\\n      try {\\n        const response = await fetch(endpoint, { method: "POST", credentials: "same-origin", headers: { "Content-Type": "application/json" }, body: JSON.stringify(data) });\\n        if (!response.ok) throw new Error("校准值保存失败");\\n        baseline = data;\\n        panel.querySelector("details").open = false;\\n        field("message").textContent = "校准值已保存到 Colab";\\n        render();\\n      } catch (error) { field("message").textContent = error.message; }\\n    };\\n    void refresh();\\n    setInterval(render, 15000);\\n    setInterval(refresh, 60000);\\n  },\\n});\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/colab_model_download/__init__.py\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'"""Download template models on the Colab server, never in the browser."""\\nimport asyncio\\nimport json\\nimport pathlib\\nimport shutil\\nimport time\\nimport uuid\\nfrom urllib.parse import urlparse\\n\\nimport requests\\nfrom aiohttp import web\\nfrom server import PromptServer\\nimport folder_paths\\n\\nWEB_DIRECTORY = "./web"\\nNODE_CLASS_MAPPINGS = {}\\nROOT = (pathlib.Path(folder_paths.base_path) / "models").resolve()\\nCATEGORIES = {"checkpoints", "diffusion_models", "text_encoders", "clip", "unet", "vae", "loras", "controlnet", "upscale_models", "embeddings", "clip_vision", "style_models"}\\nSUFFIXES = {".safetensors", ".sft", ".gguf", ".ckpt", ".pt", ".pth", ".bin"}\\nJOBS = {}\\nSLOTS = asyncio.Semaphore(1)\\n\\n\\ndef validate(model):\\n    category = model.get("directory", "")\\n    name = model.get("name", "")\\n    url = model.get("url", "")\\n    if category not in CATEGORIES:\\n        raise ValueError("不支持的模型目录")\\n    if not name or pathlib.PurePosixPath(name).is_absolute() or ".." in pathlib.PurePosixPath(name).parts or "\\\\\\\\" in name:\\n        raise ValueError("模型文件名无效")\\n    target = (ROOT / category / name).resolve()\\n    if not target.is_relative_to(ROOT / category) or target.suffix.lower() not in SUFFIXES:\\n        raise ValueError("模型文件路径或格式无效")\\n    parsed = urlparse(url)\\n    if parsed.scheme != "https" or parsed.hostname != "huggingface.co" or "/resolve/" not in parsed.path or parsed.username or parsed.password or parsed.port not in (None, 443):\\n        raise ValueError("目前支持 Hugging Face 的 HTTPS /resolve/ 模型直链")\\n    return target\\n\\n\\ndef download(job):\\n    target = pathlib.Path(job["path"])\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    if target.is_file() and target.stat().st_size > 0:\\n        job.update(state="完成", downloaded=target.stat().st_size, total=target.stat().st_size, existing=True)\\n        return\\n    partial = target.with_name(target.name + ".part")\\n    offset = partial.stat().st_size if partial.exists() else 0\\n    headers = {"Range": f"bytes={offset}-"} if offset else {}\\n    with requests.get(job["url"], headers=headers, stream=True, timeout=(30, 120)) as response:\\n        if response.status_code not in (200, 206):\\n            raise RuntimeError(f"下载失败：HTTP {response.status_code}；受限模型需先取得访问授权")\\n        if response.status_code != 206:\\n            offset = 0\\n        remaining = int(response.headers.get("Content-Length", "0"))\\n        if remaining and shutil.disk_usage(ROOT).free < remaining + 1024**3:\\n            raise RuntimeError("Colab 磁盘空间不足，已保留至少 1 GiB 余量")\\n        job.update(state="下载中", downloaded=offset, total=offset + remaining if remaining else 0)\\n        with partial.open("ab" if offset else "wb") as output:\\n            for chunk in response.iter_content(4 * 1024**2):\\n                if chunk:\\n                    output.write(chunk)\\n                    job["downloaded"] += len(chunk)\\n        if remaining and job["downloaded"] != offset + remaining:\\n            raise RuntimeError("下载不完整；再次点击可续传")\\n    if target.suffix.lower() in (".safetensors", ".sft"):\\n        with partial.open("rb") as model_file:\\n            header_size = int.from_bytes(model_file.read(8), "little")\\n            if not 0 < header_size < min(256 * 1024**2, partial.stat().st_size - 8):\\n                raise RuntimeError("下载内容不是有效的 safetensors 模型")\\n            json.loads(model_file.read(header_size))\\n    partial.replace(target)\\n    folder_paths.filename_list_cache.clear()\\n    job.update(state="完成", total=target.stat().st_size, downloaded=target.stat().st_size)\\n\\n\\nasync def run_job(job):\\n    async with SLOTS:\\n        try:\\n            await asyncio.to_thread(download, job)\\n        except Exception as error:\\n            job.update(state="失败", error=str(error))\\n\\n\\n@PromptServer.instance.routes.post("/colab/models/download")\\nasync def enqueue(request):\\n    if request.content_type != "application/json":\\n        return web.json_response({"error": "需要 JSON 请求"}, status=415)\\n    try:\\n        models = (await request.json()).get("models", [])\\n        if not isinstance(models, list) or not 1 <= len(models) <= 30:\\n            raise ValueError("请选择 1～30 个模型")\\n        targets = [validate(model) for model in models]\\n    except (ValueError, TypeError, AttributeError) as error:\\n        return web.json_response({"error": str(error)}, status=400)\\n    ids = []\\n    for model, target in zip(models, targets):\\n        active = next((job for job in JOBS.values() if job["path"] == str(target) and job["state"] != "失败"), None)\\n        if active:\\n            ids.append(active["id"])\\n            continue\\n        job = dict(id=uuid.uuid4().hex, name=model["name"], directory=model["directory"], url=model["url"], path=str(target), state="排队中", downloaded=0, total=0, created=time.time())\\n        JOBS[job["id"]] = job\\n        ids.append(job["id"])\\n        asyncio.create_task(run_job(job))\\n    return web.json_response({"ids": ids, "destination": str(ROOT)})\\n\\n\\n@PromptServer.instance.routes.get("/colab/models/jobs")\\nasync def jobs(request):\\n    return web.json_response({"destination": str(ROOT), "jobs": [{key: value for key, value in job.items() if key != "url"} for job in JOBS.values()]})\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/colab_model_download/web/colab_model_download.js\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'import { app } from "../../scripts/app.js";\\n\\nconst singleSelector = \\\'[data-testid="missing-model-download"]\\\';\\nconst allSelector = \\\'[data-testid="missing-model-download-all"]\\\';\\n\\nfunction modelsInWorkflow() {\\n  const models = new Map();\\n  function walk(value) {\\n    if (!value || typeof value !== "object") return;\\n    if (typeof value.name === "string" && typeof value.url === "string" && typeof value.directory === "string") {\\n      models.set(value.directory + "/" + value.name, { name: value.name, url: value.url, directory: value.directory });\\n    }\\n    for (const child of Object.values(value)) walk(child);\\n  }\\n  walk(app.graph.serialize());\\n  return [...models.values()];\\n}\\n\\nfunction modelForButton(button, models) {\\n  const label = button.getAttribute("aria-label") || "";\\n  return models.find(model => label.includes(model.name));\\n}\\n\\napp.registerExtension({\\n  name: "Local.ColabModelDownloads",\\n  setup() {\\n    const panel = document.createElement("section");\\n    panel.id = "colab-model-download-status";\\n    panel.setAttribute("aria-label", "Colab 模型下载进度");\\n    panel.style.cssText = "display:none;position:fixed;left:76px;bottom:72px;z-index:1001;width:440px;max-height:280px;overflow:auto;padding:14px;border:1px solid #666;border-radius:8px;background:#202020;color:#eee;font:12px/1.6 sans-serif;box-shadow:0 3px 12px #0008";\\n    document.body.appendChild(panel);\\n    const refreshed = new Set();\\n    let polling = false;\\n    function message(text) { panel.style.display = "block"; panel.textContent = text; }\\n    function bytes(n) { return (n / 1024**3).toFixed(2) + " GiB"; }\\n    async function poll() {\\n      if (polling) return;\\n      polling = true;\\n      try {\\n        const response = await fetch("/colab/models/jobs", { credentials: "same-origin" });\\n        if (!response.ok) throw new Error("无法读取 Colab 下载进度");\\n        const data = await response.json();\\n        const active = data.jobs.filter(job => ["排队中", "下载中"].includes(job.state));\\n        const shown = data.jobs.slice(-12);\\n        if (shown.length) {\\n          panel.replaceChildren();\\n          panel.style.display = "block";\\n          const heading = document.createElement("strong");\\n          heading.textContent = "Colab 端模型下载";\\n          panel.appendChild(heading);\\n          const close = document.createElement("button");\\n          close.textContent = "收起";\\n          close.style.cssText = "float:right;background:none;border:0;color:#aaa;cursor:pointer";\\n          close.onclick = () => { panel.style.display = "none"; };\\n          panel.appendChild(close);\\n          for (const job of shown) {\\n            const row = document.createElement("div");\\n            row.style.cssText = "margin-top:8px;overflow-wrap:anywhere";\\n            const progress = job.total ? ` ${Math.floor(job.downloaded / job.total * 100)}% · ${bytes(job.downloaded)} / ${bytes(job.total)}` : ` ${bytes(job.downloaded)}`;\\n            row.textContent = `${job.name} — ${job.state}${progress}`;\\n            const path = document.createElement("div");\\n            path.style.color = "#aaa";\\n            path.textContent = job.error || job.path;\\n            row.appendChild(path);\\n            panel.appendChild(row);\\n            if (job.state === "完成" && !refreshed.has(job.id)) {\\n              refreshed.add(job.id);\\n              await app.refreshMissingModels({ silent: true });\\n            }\\n          }\\n        }\\n        if (active.length) setTimeout(poll, 1500);\\n      } catch (error) { message(error.message); }\\n      finally { polling = false; }\\n    }\\n    async function submit(models) {\\n      if (!models.length) throw new Error("工作流中找不到模型下载地址；已阻止浏览器本地下载。");\\n      message("正在提交到 Colab；文件将写入 /content/ComfyUI/models/ …");\\n      const response = await fetch("/colab/models/download", {\\n        method: "POST", credentials: "same-origin", headers: { "Content-Type": "application/json" }, body: JSON.stringify({ models }),\\n      });\\n      const result = await response.json();\\n      if (!response.ok) throw new Error(result.error || "Colab 下载提交失败");\\n      await poll();\\n    }\\n    document.addEventListener("click", event => {\\n      const button = event.target.closest?.(`${singleSelector},${allSelector}`);\\n      if (!button) return;\\n      event.preventDefault();\\n      event.stopImmediatePropagation();\\n      try {\\n        const available = modelsInWorkflow();\\n        const selected = button.matches(singleSelector)\\n          ? [modelForButton(button, available)].filter(Boolean)\\n          : [...document.querySelectorAll(singleSelector)].map(b => modelForButton(b, available)).filter(Boolean);\\n        void submit(selected).catch(error => message(error.message));\\n      } catch (error) { message(error.message); }\\n    }, true);\\n    function relabel() {\\n      for (const button of document.querySelectorAll(singleSelector)) {\\n        button.title = "下载至 Colab 的模型目录，不保存到本机";\\n        const label = button.getAttribute("aria-label");\\n        if (label?.startsWith("下载 ")) button.setAttribute("aria-label", label.replace("下载 ", "下载到 Colab "));\\n        for (const child of button.childNodes) {\\n          if (child.nodeType === Node.TEXT_NODE && ["下载", "Download"].includes(child.textContent.trim())) child.textContent = "下载到 Colab";\\n        }\\n      }\\n      for (const button of document.querySelectorAll(allSelector)) {\\n        const text = button.querySelector("span.truncate");\\n        if (text && !text.textContent.includes("Colab")) text.textContent = text.textContent.replace("全部下载", "全部下载到 Colab").replace("Download All", "Download All to Colab");\\n        button.title = "全部下载至 Colab，不通过浏览器下载到本机";\\n      }\\n    }\\n    new MutationObserver(relabel).observe(document.body, { childList: true, subtree: true });\\n    relabel();\\n    void poll();\\n  },\\n});\\n\')\n# H3 参考图编辑节点和工作流\nfrom pathlib import Path\nimport json\nimport shutil\nfrom PIL import Image, ImageDraw\nROOT = Path(\'/content/ComfyUI\')\np = Path(\'/content/ComfyUI/custom_nodes/h3_reference_edit/__init__.py\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'"""Reference-role controls for ComfyUI\\\'s native MiniMax H3 Image node."""\\nfrom comfy_extras.nodes_minimax_h3 import MiniMaxH3ImageConditioning\\n\\n\\nREFERENCE_ROLES = (\\n    (\\\'face\\\', \\\'换脸\\\', \\\'Replace the main subject\\\\\\\'s face using the facial identity in {picture}. Match the expression, skin detail and lighting naturally.\\\'),\\n    (\\\'style\\\', \\\'造型\\\', \\\'Use the hairstyle, makeup, body pose and styling in {picture} for the main subject.\\\'),\\n    (\\\'clothing\\\', \\\'换衣\\\', \\\'Dress the main subject in the outfit from {picture}. Reproduce the garment design, colors, fabric and fit.\\\'),\\n    (\\\'background\\\', \\\'背景\\\', \\\'Replace the background with the setting in {picture}. Integrate the main subject with the scene perspective, lighting and contact shadows.\\\'),\\n)\\n\\n\\nclass H3ReferenceEdit:\\n    @classmethod\\n    def INPUT_TYPES(cls):\\n        required = {\\n            \\\'clip\\\': (\\\'CLIP\\\',),\\n            \\\'vae\\\': (\\\'VAE\\\',),\\n            \\\'instructions\\\': (\\\'STRING\\\', {\\\'multiline\\\': True, \\\'default\\\': \\\'Produce a coherent edit with natural lighting and matching perspective.\\\'}),\\n            \\\'width\\\': (\\\'INT\\\', {\\\'default\\\': 2048, \\\'min\\\': 512, \\\'max\\\': 8192, \\\'step\\\': 32}),\\n            \\\'height\\\': (\\\'INT\\\', {\\\'default\\\': 2048, \\\'min\\\': 512, \\\'max\\\': 8192, \\\'step\\\': 32}),\\n        }\\n        for role, label, _ in REFERENCE_ROLES:\\n            required[role + \\\'_enabled\\\'] = (\\\'BOOLEAN\\\', {\\\'default\\\': False, \\\'label_on\\\': label + \\\' 开\\\', \\\'label_off\\\': label + \\\' 关\\\'})\\n        optional = {\\\'base_image\\\': (\\\'IMAGE\\\', {\\\'lazy\\\': True})}\\n        optional.update({role + \\\'_reference\\\': (\\\'IMAGE\\\', {\\\'lazy\\\': True}) for role, _, _ in REFERENCE_ROLES})\\n        return {\\\'required\\\': required, \\\'optional\\\': optional}\\n\\n    RETURN_TYPES = (\\\'CONDITIONING\\\', \\\'LATENT\\\', \\\'STRING\\\')\\n    RETURN_NAMES = (\\\'positive\\\', \\\'latent\\\', \\\'reference_prompt\\\')\\n    FUNCTION = \\\'encode\\\'\\n    CATEGORY = \\\'MiniMax H3/Reference Edit\\\'\\n    DESCRIPTION = \\\'可选原图和人脸、造型、衣服、背景参考图。开启对应开关前先上传参考图。\\\'\\n\\n    def check_lazy_status(self, **inputs):\\n        required = [\\\'base_image\\\'] if \\\'base_image\\\' in inputs and inputs[\\\'base_image\\\'] is None else []\\n        return required + [\\n            role + \\\'_reference\\\'\\n            for role, _, _ in REFERENCE_ROLES\\n            if inputs.get(role + \\\'_enabled\\\') and role + \\\'_reference\\\' in inputs and inputs[role + \\\'_reference\\\'] is None\\n        ]\\n\\n    def encode(self, clip, vae, instructions, width, height, base_image=None, **inputs):\\n        references = {}\\n        if base_image is not None:\\n            references[\\\'image_1\\\'] = base_image\\n            directions = [\\n                \\\'Use <Picture 1> as the base image to edit.\\\',\\n                \\\'Keep the composition and details consistent with the base image while applying the requested edits.\\\',\\n            ]\\n        else:\\n            directions = [\\\'Generate a new image using the following instructions and enabled references.\\\']\\n        for role, label, direction in REFERENCE_ROLES:\\n            if not inputs.get(role + \\\'_enabled\\\'):\\n                continue\\n            reference = inputs.get(role + \\\'_reference\\\')\\n            if reference is None:\\n                raise ValueError(f\\\'已启用{label}，请上传并连接对应参考图。\\\')\\n            number = len(references) + 1\\n            references[f\\\'image_{number}\\\'] = reference\\n            directions.append(direction.format(picture=f\\\'<Picture {number}>\\\'))\\n        if inputs.get(\\\'face_enabled\\\') and inputs.get(\\\'style_enabled\\\'):\\n            directions.append(\\\'Use the face reference for facial identity and the styling reference for hairstyle, makeup and pose.\\\')\\n        if inputs.get(\\\'style_enabled\\\') and inputs.get(\\\'clothing_enabled\\\'):\\n            directions.append(\\\'Use the clothing reference for the outfit and the styling reference for hairstyle, makeup and pose.\\\')\\n        if instructions.strip():\\n            directions.append(\\\'Additional editing instructions: \\\' + instructions.strip())\\n        if not references and not instructions.strip():\\n            raise ValueError(\\\'请填写生成指令或提供至少一张启用的参考图。\\\')\\n        prompt = \\\'\\\\n\\\'.join(directions)\\n        output = MiniMaxH3ImageConditioning.execute(\\n            clip=clip, vae=vae, prompt=prompt, width=width, height=height, images=references\\n        )\\n        positive, latent = output.result\\n        return positive, latent, prompt\\n\\n\\nNODE_CLASS_MAPPINGS = {\\\'H3ReferenceEdit\\\': H3ReferenceEdit}\\nNODE_DISPLAY_NAME_MAPPINGS = {\\\'H3ReferenceEdit\\\': \\\'H3 参考图编辑 · 换脸 / 造型 / 换衣 / 背景\\\'}\\nWEB_DIRECTORY = \\\'./web\\\'\\n\\n# Register result routes when ComfyUI imports this package.\\nfrom . import result_routes\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/h3_reference_edit/result_routes.py\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text("from pathlib import Path\\nfrom aiohttp import web\\nimport folder_paths\\nfrom server import PromptServer\\nfrom .result_files import ResultFiles\\n\\nstore = ResultFiles(folder_paths.get_output_directory(), Path(folder_paths.get_user_directory()) / \'default/h3_studio_trash\')\\nroutes = PromptServer.instance.routes\\n\\n\\n@routes.get(\'/h3-studio/results\')\\nasync def list_results(request):\\n    return web.json_response({\'items\': store.results()})\\n\\n\\n@routes.get(\'/h3-studio/results/trash\')\\nasync def list_trash(request):\\n    return web.json_response({\'items\': store.items()})\\n\\n\\n@routes.post(\'/h3-studio/results/trash\')\\nasync def trash_result(request):\\n    try:\\n        payload = await request.json()\\n        return web.json_response(store.trash(payload.get(\'image\')))\\n    except FileNotFoundError as error:\\n        return web.json_response({\'error\': str(error)}, status=404)\\n    except ValueError as error:\\n        return web.json_response({\'error\': str(error)}, status=400)\\n\\n\\n@routes.post(\'/h3-studio/results/restore\')\\nasync def restore_result(request):\\n    try:\\n        payload = await request.json()\\n        return web.json_response(store.restore(payload.get(\'token\')))\\n    except FileExistsError as error:\\n        return web.json_response({\'error\': str(error)}, status=409)\\n    except FileNotFoundError:\\n        return web.json_response({\'error\': \'回收站记录不存在。\'}, status=404)\\n    except ValueError as error:\\n        return web.json_response({\'error\': str(error)}, status=400)\\n")\np = Path(\'/content/ComfyUI/custom_nodes/h3_reference_edit/result_files.py\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text(\'"""Recoverable result deletion, confined to ComfyUI\\\'s output directory."""\\nfrom pathlib import Path\\nimport json\\nimport threading\\nimport time\\nimport uuid\\n\\n\\nclass ResultFiles:\\n    def __init__(self, output_root, trash_root):\\n        self.output_root = Path(output_root).resolve()\\n        self.trash_root = Path(trash_root)\\n        self.lock = threading.Lock()\\n\\n    def destination(self, image):\\n        if not isinstance(image, dict) or image.get(\\\'type\\\', \\\'output\\\') != \\\'output\\\':\\n            raise ValueError(\\\'只能删除输出图片。\\\')\\n        filename = image.get(\\\'filename\\\')\\n        if not isinstance(filename, str) or not filename or Path(filename).name != filename or \\\'\\\\\\\\\\\' in filename:\\n            raise ValueError(\\\'图片文件名无效。\\\')\\n        if Path(filename).suffix.lower() not in [\\\'.png\\\', \\\'.jpg\\\', \\\'.jpeg\\\', \\\'.webp\\\']:\\n            raise ValueError(\\\'只支持删除图片输出。\\\')\\n        subfolder = image.get(\\\'subfolder\\\', \\\'\\\')\\n        if not isinstance(subfolder, str):\\n            raise ValueError(\\\'图片目录无效。\\\')\\n        candidate = self.output_root / subfolder / filename\\n        if candidate.is_symlink():\\n            raise ValueError(\\\'不支持删除符号链接。\\\')\\n        path = candidate.resolve()\\n        if not path.is_relative_to(self.output_root):\\n            raise ValueError(\\\'图片不在输出目录中。\\\')\\n        return path\\n\\n    def results(self):\\n        results = []\\n        for path in self.output_root.rglob(\\\'*\\\'):\\n            if not path.is_file() or path.is_symlink() or path.name.startswith(\\\'_H3Studio_selftest_\\\'):\\n                continue\\n            if path.suffix.lower() not in [\\\'.png\\\', \\\'.jpg\\\', \\\'.jpeg\\\', \\\'.webp\\\']:\\n                continue\\n            resolved = path.resolve()\\n            if not resolved.is_relative_to(self.output_root):\\n                continue\\n            relative = resolved.relative_to(self.output_root)\\n            subfolder = relative.parent.as_posix()\\n            results.append({\\\'image\\\': {\\\'filename\\\': relative.name, \\\'subfolder\\\': \\\'\\\' if subfolder == \\\'.\\\' else subfolder, \\\'type\\\': \\\'output\\\'}, \\\'modified_at_ms\\\': resolved.stat().st_mtime_ns // 1000000})\\n        return sorted(results, key=lambda item: item[\\\'modified_at_ms\\\'], reverse=True)[:100]\\n\\n    def items(self):\\n        if not self.trash_root.exists():\\n            return []\\n        results = []\\n        for manifest in self.trash_root.glob(\\\'*/manifest.json\\\'):\\n            if (manifest.parent / \\\'image\\\').is_file():\\n                results.append(json.loads(manifest.read_text()))\\n        return sorted(results, key=lambda item: item[\\\'deleted_at_ms\\\'], reverse=True)\\n\\n    def trash(self, image):\\n        with self.lock:\\n            source = self.destination(image)\\n            if not source.is_file():\\n                raise FileNotFoundError(\\\'图片已经删除或文件不存在。\\\')\\n            token = str(uuid.uuid4())\\n            directory = self.trash_root / token\\n            directory.mkdir(parents=True, mode=0o700)\\n            item = {\\\'token\\\': token, \\\'image\\\': {\\\'filename\\\': image[\\\'filename\\\'], \\\'subfolder\\\': image.get(\\\'subfolder\\\', \\\'\\\'), \\\'type\\\': \\\'output\\\'}, \\\'deleted_at_ms\\\': round(time.time() * 1000)}\\n            manifest = directory / \\\'manifest.json\\\'\\n            try:\\n                manifest.write_text(json.dumps(item, ensure_ascii=False))\\n                source.replace(directory / \\\'image\\\')\\n            except Exception:\\n                manifest.unlink(missing_ok=True)\\n                directory.rmdir()\\n                raise\\n            return item\\n\\n    def restore(self, token):\\n        token = str(uuid.UUID(str(token)))\\n        with self.lock:\\n            directory = self.trash_root / token\\n            item = json.loads((directory / \\\'manifest.json\\\').read_text())\\n            target = self.destination(item[\\\'image\\\'])\\n            if target.exists():\\n                raise FileExistsError(\\\'原位置存在同名文件，未覆盖现有图片。\\\')\\n            target.parent.mkdir(parents=True, exist_ok=True)\\n            (directory / \\\'image\\\').replace(target)\\n            (directory / \\\'manifest.json\\\').unlink()\\n            directory.rmdir()\\n            return item\\n\')\np = Path(\'/content/ComfyUI/custom_nodes/h3_reference_edit/web/labels.js\')\np.parent.mkdir(parents=True, exist_ok=True)\np.write_text("import { app } from \'../../scripts/app.js\';\\nconst labels = {\\n  instructions: \'补充编辑指令\', width: \'输出宽度\', height: \'输出高度\',\\n  face_enabled: \'启用换脸\', style_enabled: \'启用造型 / 姿态\',\\n  clothing_enabled: \'启用换衣\', background_enabled: \'启用背景\',\\n};\\napp.registerExtension({\\n  name: \'Local.H3ReferenceEditLabels\',\\n  nodeCreated(node) {\\n    if ((node.comfyClass ?? node.constructor.comfyClass ?? node.type) !== \'H3ReferenceEdit\') return;\\n    for (const widget of node.widgets ?? []) {\\n      if (labels[widget.name]) widget.label = labels[widget.name];\\n    }\\n  },\\n});\\n")\nh3_edit_workflow = {\'last_node_id\': 17, \'last_link_id\': 18, \'nodes\': [{\'id\': 1, \'type\': \'UNETLoader\', \'pos\': [0, 500], \'size\': [440, 90], \'flags\': {}, \'order\': 0, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'MODEL\', \'name\': \'MODEL\', \'type\': \'MODEL\', \'links\': [9]}], \'properties\': {\'Node name for S&R\': \'UNETLoader\', \'models\': [{\'name\': \'minimax_h3_image_pruned_w4a8.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/diffusion_models/minimax_h3_image_pruned_w4a8.safetensors\', \'directory\': \'diffusion_models\'}]}, \'widgets_values\': [\'minimax_h3_image_pruned_w4a8.safetensors\', \'default\'], \'widgets_values_named\': {\'unet_name\': \'minimax_h3_image_pruned_w4a8.safetensors\', \'weight_dtype\': \'default\'}}, {\'id\': 2, \'type\': \'CLIPLoader\', \'pos\': [0, 650], \'size\': [440, 120], \'flags\': {}, \'order\': 1, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'CLIP\', \'name\': \'CLIP\', \'type\': \'CLIP\', \'links\': [1]}], \'properties\': {\'Node name for S&R\': \'CLIPLoader\', \'models\': [{\'name\': \'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'directory\': \'text_encoders\'}]}, \'widgets_values\': [\'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'minimax\', \'default\'], \'widgets_values_named\': {\'clip_name\': \'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'type\': \'minimax\', \'device\': \'default\'}}, {\'id\': 3, \'type\': \'VAELoader\', \'pos\': [0, 840], \'size\': [440, 70], \'flags\': {}, \'order\': 2, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'VAE\', \'name\': \'VAE\', \'type\': \'VAE\', \'links\': [2, 14]}], \'properties\': {\'Node name for S&R\': \'VAELoader\', \'models\': [{\'name\': \'minimax_h3_image_vae_fp16.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors\', \'directory\': \'vae\'}]}, \'widgets_values\': [\'minimax_h3_image_vae_fp16.safetensors\'], \'widgets_values_named\': {\'vae_name\': \'minimax_h3_image_vae_fp16.safetensors\'}}, {\'id\': 5, \'type\': \'H3ReferenceEdit\', \'pos\': [520, 500], \'size\': [540, 530], \'flags\': {}, \'order\': 3, \'mode\': 0, \'inputs\': [{\'name\': \'clip\', \'type\': \'CLIP\', \'link\': 1}, {\'name\': \'vae\', \'type\': \'VAE\', \'link\': 2}, {\'name\': \'base_image\', \'type\': \'IMAGE\', \'shape\': 7, \'link\': 3}, {\'name\': \'face_reference\', \'type\': \'IMAGE\', \'shape\': 7, \'link\': 4}, {\'name\': \'style_reference\', \'type\': \'IMAGE\', \'shape\': 7, \'link\': 5}, {\'name\': \'clothing_reference\', \'type\': \'IMAGE\', \'shape\': 7, \'link\': 6}, {\'name\': \'background_reference\', \'type\': \'IMAGE\', \'shape\': 7, \'link\': 7}], \'outputs\': [{\'name\': \'positive\', \'type\': \'CONDITIONING\', \'links\': [8, 10]}, {\'name\': \'latent\', \'type\': \'LATENT\', \'links\': [12]}, {\'name\': \'reference_prompt\', \'type\': \'STRING\', \'links\': [18]}], \'properties\': {\'Node name for S&R\': \'H3ReferenceEdit\'}, \'widgets_values\': [\'Produce a coherent edit with natural lighting and matching perspective.\', 2048, 2048, False, False, False, False], \'widgets_values_named\': {\'instructions\': \'Produce a coherent edit with natural lighting and matching perspective.\', \'width\': 2048, \'height\': 2048, \'face_enabled\': False, \'style_enabled\': False, \'clothing_enabled\': False, \'background_enabled\': False}, \'title\': \'编辑控制 · 先上传参考图，再启用对应开关\'}, {\'id\': 6, \'type\': \'ConditioningZeroOut\', \'pos\': [520, 1080], \'size\': [300, 50], \'flags\': {}, \'order\': 4, \'mode\': 0, \'inputs\': [{\'localized_name\': \'conditioning\', \'name\': \'conditioning\', \'type\': \'CONDITIONING\', \'link\': 8}], \'outputs\': [{\'localized_name\': \'CONDITIONING\', \'name\': \'CONDITIONING\', \'type\': \'CONDITIONING\', \'links\': [11]}], \'properties\': {\'Node name for S&R\': \'ConditioningZeroOut\'}}, {\'id\': 7, \'type\': \'KSampler\', \'pos\': [1130, 500], \'size\': [320, 480], \'flags\': {}, \'order\': 5, \'mode\': 0, \'inputs\': [{\'localized_name\': \'model\', \'name\': \'model\', \'type\': \'MODEL\', \'link\': 9}, {\'localized_name\': \'positive\', \'name\': \'positive\', \'type\': \'CONDITIONING\', \'link\': 10}, {\'localized_name\': \'negative\', \'name\': \'negative\', \'type\': \'CONDITIONING\', \'link\': 11}, {\'localized_name\': \'latent_image\', \'name\': \'latent_image\', \'type\': \'LATENT\', \'link\': 12}, {\'localized_name\': \'seed\', \'name\': \'seed\', \'type\': \'INT\', \'widget\': {\'name\': \'seed\'}, \'link\': None}, {\'localized_name\': \'sampler_name\', \'name\': \'sampler_name\', \'type\': \'COMBO\', \'widget\': {\'name\': \'sampler_name\'}, \'link\': None}, {\'localized_name\': \'scheduler\', \'name\': \'scheduler\', \'type\': \'COMBO\', \'widget\': {\'name\': \'scheduler\'}, \'link\': None}], \'outputs\': [{\'localized_name\': \'LATENT\', \'name\': \'LATENT\', \'type\': \'LATENT\', \'links\': [13]}], \'properties\': {\'Node name for S&R\': \'KSampler\'}, \'widgets_values\': [42, \'randomize\', 6, 1, \'euler\', \'simple\', 1], \'widgets_values_named\': {\'seed\': 42, \'control_after_generate\': \'randomize\', \'steps\': 6, \'cfg\': 1, \'sampler_name\': \'euler\', \'scheduler\': \'simple\', \'denoise\': 1}}, {\'id\': 8, \'type\': \'VAEDecode\', \'pos\': [1520, 500], \'size\': [210, 50], \'flags\': {}, \'order\': 6, \'mode\': 0, \'inputs\': [{\'localized_name\': \'samples\', \'name\': \'samples\', \'type\': \'LATENT\', \'link\': 13}, {\'localized_name\': \'vae\', \'name\': \'vae\', \'type\': \'VAE\', \'link\': 14}], \'outputs\': [{\'localized_name\': \'IMAGE\', \'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [15, 17]}], \'properties\': {\'Node name for S&R\': \'VAEDecode\'}}, {\'id\': 9, \'type\': \'SaveImage\', \'pos\': [1520, 650], \'size\': [440, 480], \'flags\': {}, \'order\': 7, \'mode\': 0, \'inputs\': [{\'name\': \'images\', \'type\': \'IMAGE\', \'link\': 15}], \'outputs\': [{\'name\': \'images\', \'type\': \'IMAGE\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'SaveImage\'}, \'widgets_values\': [\'H3-Reference-Edit\'], \'widgets_values_named\': {\'filename_prefix\': \'H3-Reference-Edit\'}}, {\'id\': 10, \'type\': \'LoadImage\', \'title\': \'原图 · 可选\', \'pos\': [0, 0], \'size\': [380, 390], \'flags\': {}, \'order\': 0, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [3, 16]}, {\'name\': \'MASK\', \'type\': \'MASK\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'LoadImage\'}, \'widgets_values\': [\'H3-base-reference.png\', \'image\']}, {\'id\': 11, \'type\': \'LoadImage\', \'title\': \'人脸参考 · 换脸\', \'pos\': [420, 0], \'size\': [380, 390], \'flags\': {}, \'order\': 1, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [4]}, {\'name\': \'MASK\', \'type\': \'MASK\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'LoadImage\'}, \'widgets_values\': [\'H3-face-reference.png\', \'image\']}, {\'id\': 12, \'type\': \'LoadImage\', \'title\': \'造型参考 · 发型 / 妆容 / 姿态\', \'pos\': [840, 0], \'size\': [380, 390], \'flags\': {}, \'order\': 2, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [5]}, {\'name\': \'MASK\', \'type\': \'MASK\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'LoadImage\'}, \'widgets_values\': [\'H3-style-reference.png\', \'image\']}, {\'id\': 13, \'type\': \'LoadImage\', \'title\': \'衣服参考 · 换衣\', \'pos\': [1260, 0], \'size\': [380, 390], \'flags\': {}, \'order\': 3, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [6]}, {\'name\': \'MASK\', \'type\': \'MASK\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'LoadImage\'}, \'widgets_values\': [\'H3-clothing-reference.png\', \'image\']}, {\'id\': 14, \'type\': \'LoadImage\', \'title\': \'背景参考 · 换场景\', \'pos\': [1680, 0], \'size\': [380, 390], \'flags\': {}, \'order\': 4, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [7]}, {\'name\': \'MASK\', \'type\': \'MASK\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'LoadImage\'}, \'widgets_values\': [\'H3-background-reference.png\', \'image\']}, {\'id\': 15, \'type\': \'ImageCompare\', \'title\': \'原图 / 编辑结果 · 移动鼠标对比\', \'pos\': [2020, 500], \'size\': [500, 610], \'flags\': {}, \'order\': 20, \'mode\': 0, \'inputs\': [{\'name\': \'image_a\', \'type\': \'IMAGE\', \'link\': 16}, {\'name\': \'image_b\', \'type\': \'IMAGE\', \'link\': 17}], \'outputs\': [], \'properties\': {\'Node name for S&R\': \'ImageCompare\'}, \'widgets_values\': []}, {\'id\': 16, \'type\': \'PreviewAny\', \'title\': \'实际参考图指令\', \'pos\': [520, 1200], \'size\': [540, 260], \'flags\': {}, \'order\': 21, \'mode\': 0, \'inputs\': [{\'name\': \'source\', \'type\': \'*\', \'link\': 18}], \'outputs\': [], \'properties\': {\'Node name for S&R\': \'PreviewAny\'}, \'widgets_values\': []}, {\'id\': 17, \'type\': \'MarkdownNote\', \'pos\': [0, 1010], \'size\': [440, 400], \'flags\': {}, \'order\': 22, \'mode\': 0, \'inputs\': [], \'outputs\': [], \'properties\': {}, \'widgets_values\': [\'## 参考图编辑\\n\\n1. 有原图时编辑原图；没有原图时断开原图连线，按提示词和参考图生成。\\n2. 上传需要的参考图，再开启编辑控制中的对应开关。可同时开启多项。\\n3. 在补充指令中描述细节，点击运行。\\n\\n人脸参考决定面部身份；造型参考决定发型、妆容和姿态；衣服参考决定服装；背景参考决定场景。\\n\\n人脸为生成式参考编辑，相似度需用实际照片检查。默认 2048×2048、6 步、CFG=1。\']}], \'links\': [[1, 2, 0, 5, 0, \'CLIP\'], [2, 3, 0, 5, 1, \'VAE\'], [3, 10, 0, 5, 2, \'IMAGE\'], [4, 11, 0, 5, 3, \'IMAGE\'], [5, 12, 0, 5, 4, \'IMAGE\'], [6, 13, 0, 5, 5, \'IMAGE\'], [7, 14, 0, 5, 6, \'IMAGE\'], [8, 5, 0, 6, 0, \'CONDITIONING\'], [9, 1, 0, 7, 0, \'MODEL\'], [10, 5, 0, 7, 1, \'CONDITIONING\'], [11, 6, 0, 7, 2, \'CONDITIONING\'], [12, 5, 1, 7, 3, \'LATENT\'], [13, 7, 0, 8, 0, \'LATENT\'], [14, 3, 0, 8, 1, \'VAE\'], [15, 8, 0, 9, 0, \'IMAGE\'], [16, 10, 0, 15, 0, \'IMAGE\'], [17, 8, 0, 15, 1, \'IMAGE\'], [18, 5, 2, 16, 0, \'STRING\']], \'groups\': [], \'config\': {}, \'extra\': {\'ds\': {\'scale\': 0.48, \'offset\': [70, 100]}}, \'version\': 0.4}\nworkflow_dir = ROOT / \'user/default/workflows\'\nworkflow_dir.mkdir(parents=True, exist_ok=True)\n(workflow_dir / \'MiniMax-H3-Image-Reference-Edit-L4.json\').write_text(json.dumps(h3_edit_workflow, ensure_ascii=False, indent=2))\ninput_dir = ROOT / \'input\'\ninput_dir.mkdir(parents=True, exist_ok=True)\nbase_demo = input_dir / \'H3-base-reference.png\'\nif not base_demo.exists():\n    result_file = Path(\'/content/H3Image-test-result.json\')\n    if result_file.exists():\n        shutil.copyfile(json.loads(result_file.read_text())[\'filename\'], base_demo)\n    else:\n        Image.new(\'RGB\', (512,512), \'#ddd7ce\').save(base_demo)\nfor role in [\'face\',\'style\',\'clothing\',\'background\']:\n    target = input_dir / f\'H3-{role}-reference.png\'\n    if not target.exists():\n        image = Image.new(\'RGB\', (512,512), \'#dedbd5\')\n        ImageDraw.Draw(image).text((40,240), f\'Upload {role} reference\', fill=\'#615b52\')\n        image.save(target)\n\nworkflow_dir = ROOT / \'user/default/workflows\'\nworkflow_dir.mkdir(parents=True, exist_ok=True)\nh3_workflow = {\'last_node_id\': 9, \'last_link_id\': 10, \'nodes\': [{\'id\': 1, \'type\': \'UNETLoader\', \'pos\': [0, 0], \'size\': [590, 90], \'flags\': {}, \'order\': 0, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'MODEL\', \'name\': \'MODEL\', \'type\': \'MODEL\', \'links\': [4]}], \'properties\': {\'Node name for S&R\': \'UNETLoader\', \'models\': [{\'name\': \'minimax_h3_image_pruned_w4a8.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/diffusion_models/minimax_h3_image_pruned_w4a8.safetensors\', \'directory\': \'diffusion_models\'}]}, \'widgets_values\': [\'minimax_h3_image_pruned_w4a8.safetensors\', \'default\'], \'widgets_values_named\': {\'unet_name\': \'minimax_h3_image_pruned_w4a8.safetensors\', \'weight_dtype\': \'default\'}}, {\'id\': 2, \'type\': \'CLIPLoader\', \'pos\': [0, 160], \'size\': [590, 120], \'flags\': {}, \'order\': 1, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'CLIP\', \'name\': \'CLIP\', \'type\': \'CLIP\', \'links\': [1]}], \'properties\': {\'Node name for S&R\': \'CLIPLoader\', \'models\': [{\'name\': \'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'directory\': \'text_encoders\'}]}, \'widgets_values\': [\'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'minimax\', \'default\'], \'widgets_values_named\': {\'clip_name\': \'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'type\': \'minimax\', \'device\': \'default\'}}, {\'id\': 3, \'type\': \'VAELoader\', \'pos\': [0, 350], \'size\': [590, 70], \'flags\': {}, \'order\': 2, \'mode\': 0, \'inputs\': [], \'outputs\': [{\'localized_name\': \'VAE\', \'name\': \'VAE\', \'type\': \'VAE\', \'links\': [2, 9]}], \'properties\': {\'Node name for S&R\': \'VAELoader\', \'models\': [{\'name\': \'minimax_h3_image_vae_fp16.safetensors\', \'url\': \'https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors\', \'directory\': \'vae\'}]}, \'widgets_values\': [\'minimax_h3_image_vae_fp16.safetensors\'], \'widgets_values_named\': {\'vae_name\': \'minimax_h3_image_vae_fp16.safetensors\'}}, {\'id\': 5, \'type\': \'MiniMaxH3ImageConditioning\', \'pos\': [650, 0], \'size\': [440, 400], \'flags\': {}, \'order\': 3, \'mode\': 0, \'inputs\': [{\'localized_name\': \'clip\', \'name\': \'clip\', \'type\': \'CLIP\', \'link\': 1}, {\'localized_name\': \'vae\', \'name\': \'vae\', \'shape\': 7, \'type\': \'VAE\', \'link\': 2}, {\'localized_name\': \'image_1\', \'name\': \'images.image_1\', \'shape\': 7, \'type\': \'IMAGE\', \'link\': None}, {\'localized_name\': \'prompt\', \'name\': \'prompt\', \'type\': \'STRING\', \'widget\': {\'name\': \'prompt\'}, \'link\': None}, {\'localized_name\': \'width\', \'name\': \'width\', \'type\': \'INT\', \'widget\': {\'name\': \'width\'}, \'link\': None}, {\'localized_name\': \'height\', \'name\': \'height\', \'type\': \'INT\', \'widget\': {\'name\': \'height\'}, \'link\': None}], \'outputs\': [{\'localized_name\': \'positive\', \'name\': \'positive\', \'type\': \'CONDITIONING\', \'links\': [3, 5]}, {\'localized_name\': \'LATENT\', \'name\': \'LATENT\', \'type\': \'LATENT\', \'links\': [7]}], \'properties\': {\'Node name for S&R\': \'MiniMaxH3ImageConditioning\'}, \'widgets_values\': [\'A cinematic editorial photograph of a small red fox standing in a Japanese garden after rain, wet moss-covered stones, delicate maple leaves, soft morning mist, warm sunlight filtering through trees, richly detailed realistic fur, natural colors, shallow depth of field. A tasteful wooden sign in the background reads "MINIMAX H3" in clear lettering. Peaceful composition, professional wildlife photography.\', 2048, 2048], \'widgets_values_named\': {\'prompt\': \'A cinematic editorial photograph of a small red fox standing in a Japanese garden after rain, wet moss-covered stones, delicate maple leaves, soft morning mist, warm sunlight filtering through trees, richly detailed realistic fur, natural colors, shallow depth of field. A tasteful wooden sign in the background reads "MINIMAX H3" in clear lettering. Peaceful composition, professional wildlife photography.\', \'width\': 2048, \'height\': 2048}}, {\'id\': 6, \'type\': \'ConditioningZeroOut\', \'pos\': [650, 500], \'size\': [300, 50], \'flags\': {}, \'order\': 4, \'mode\': 0, \'inputs\': [{\'localized_name\': \'conditioning\', \'name\': \'conditioning\', \'type\': \'CONDITIONING\', \'link\': 3}], \'outputs\': [{\'localized_name\': \'CONDITIONING\', \'name\': \'CONDITIONING\', \'type\': \'CONDITIONING\', \'links\': [6]}], \'properties\': {\'Node name for S&R\': \'ConditioningZeroOut\'}}, {\'id\': 7, \'type\': \'KSampler\', \'pos\': [1180, 0], \'size\': [320, 480], \'flags\': {}, \'order\': 5, \'mode\': 0, \'inputs\': [{\'localized_name\': \'model\', \'name\': \'model\', \'type\': \'MODEL\', \'link\': 4}, {\'localized_name\': \'positive\', \'name\': \'positive\', \'type\': \'CONDITIONING\', \'link\': 5}, {\'localized_name\': \'negative\', \'name\': \'negative\', \'type\': \'CONDITIONING\', \'link\': 6}, {\'localized_name\': \'latent_image\', \'name\': \'latent_image\', \'type\': \'LATENT\', \'link\': 7}, {\'localized_name\': \'seed\', \'name\': \'seed\', \'type\': \'INT\', \'widget\': {\'name\': \'seed\'}, \'link\': None}, {\'localized_name\': \'sampler_name\', \'name\': \'sampler_name\', \'type\': \'COMBO\', \'widget\': {\'name\': \'sampler_name\'}, \'link\': None}, {\'localized_name\': \'scheduler\', \'name\': \'scheduler\', \'type\': \'COMBO\', \'widget\': {\'name\': \'scheduler\'}, \'link\': None}], \'outputs\': [{\'localized_name\': \'LATENT\', \'name\': \'LATENT\', \'type\': \'LATENT\', \'links\': [8]}], \'properties\': {\'Node name for S&R\': \'KSampler\'}, \'widgets_values\': [42, \'fixed\', 6, 1, \'euler\', \'simple\', 1], \'widgets_values_named\': {\'seed\': 42, \'control_after_generate\': \'fixed\', \'steps\': 6, \'cfg\': 1, \'sampler_name\': \'euler\', \'scheduler\': \'simple\', \'denoise\': 1}}, {\'id\': 8, \'type\': \'VAEDecode\', \'pos\': [1580, 0], \'size\': [210, 50], \'flags\': {}, \'order\': 6, \'mode\': 0, \'inputs\': [{\'localized_name\': \'samples\', \'name\': \'samples\', \'type\': \'LATENT\', \'link\': 8}, {\'localized_name\': \'vae\', \'name\': \'vae\', \'type\': \'VAE\', \'link\': 9}], \'outputs\': [{\'localized_name\': \'IMAGE\', \'name\': \'IMAGE\', \'type\': \'IMAGE\', \'links\': [10]}], \'properties\': {\'Node name for S&R\': \'VAEDecode\'}}, {\'id\': 9, \'type\': \'SaveImage\', \'pos\': [1580, 150], \'size\': [560, 540], \'flags\': {}, \'order\': 7, \'mode\': 0, \'inputs\': [{\'name\': \'images\', \'type\': \'IMAGE\', \'link\': 10}], \'outputs\': [{\'name\': \'images\', \'type\': \'IMAGE\', \'links\': []}], \'properties\': {\'Node name for S&R\': \'SaveImage\'}, \'widgets_values\': [\'MiniMax-H3-Image\'], \'widgets_values_named\': {\'filename_prefix\': \'MiniMax-H3-Image\'}}], \'links\': [[1, 2, 0, 5, 0, \'CLIP\'], [2, 3, 0, 5, 1, \'VAE\'], [3, 5, 0, 6, 0, \'CONDITIONING\'], [4, 1, 0, 7, 0, \'MODEL\'], [5, 5, 0, 7, 1, \'CONDITIONING\'], [6, 6, 0, 7, 2, \'CONDITIONING\'], [7, 5, 1, 7, 3, \'LATENT\'], [8, 7, 0, 8, 0, \'LATENT\'], [9, 3, 0, 8, 1, \'VAE\'], [10, 8, 0, 9, 0, \'IMAGE\']], \'groups\': [], \'config\': {}, \'extra\': {\'ds\': {\'scale\': 0.6, \'offset\': [70, 100]}}, \'version\': 0.4}\nh3_prompt = {\'1\': {\'class_type\': \'UNETLoader\', \'inputs\': {\'unet_name\': \'minimax_h3_image_pruned_w4a8.safetensors\', \'weight_dtype\': \'default\'}}, \'2\': {\'class_type\': \'CLIPLoader\', \'inputs\': {\'clip_name\': \'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors\', \'type\': \'minimax\', \'device\': \'default\'}}, \'3\': {\'class_type\': \'VAELoader\', \'inputs\': {\'vae_name\': \'minimax_h3_image_vae_fp16.safetensors\'}}, \'5\': {\'class_type\': \'MiniMaxH3ImageConditioning\', \'inputs\': {\'clip\': [\'2\', 0], \'vae\': [\'3\', 0], \'prompt\': \'A cinematic editorial photograph of a small red fox standing in a Japanese garden after rain, wet moss-covered stones, delicate maple leaves, soft morning mist, warm sunlight filtering through trees, richly detailed realistic fur, natural colors, shallow depth of field. A tasteful wooden sign in the background reads "MINIMAX H3" in clear lettering. Peaceful composition, professional wildlife photography.\', \'width\': 2048, \'height\': 2048}}, \'6\': {\'class_type\': \'ConditioningZeroOut\', \'inputs\': {\'conditioning\': [\'5\', 0]}}, \'7\': {\'class_type\': \'KSampler\', \'inputs\': {\'model\': [\'1\', 0], \'positive\': [\'5\', 0], \'negative\': [\'6\', 0], \'latent_image\': [\'5\', 1], \'seed\': 42, \'steps\': 6, \'cfg\': 1, \'sampler_name\': \'euler\', \'scheduler\': \'simple\', \'denoise\': 1}}, \'8\': {\'class_type\': \'VAEDecode\', \'inputs\': {\'samples\': [\'7\', 0], \'vae\': [\'3\', 0]}}, \'9\': {\'class_type\': \'SaveImage\', \'inputs\': {\'images\': [\'8\', 0], \'filename_prefix\': \'MiniMax-H3-Image\'}}}\n(workflow_dir / \'MiniMax-H3-Image-L4.json\').write_text(json.dumps(h3_workflow, ensure_ascii=False, indent=2))\n(ROOT / \'user/default/MiniMax-H3-Image-L4-API.json\').write_text(json.dumps(h3_prompt, indent=2))\nif \'comfy_process\' in globals() and comfy_process.poll() is None:\n    pending = requests.get(\'http://127.0.0.1:8188/queue\', timeout=10).json()\n    if pending.get(\'queue_running\') or pending.get(\'queue_pending\'):\n        raise RuntimeError(\'ComfyUI 有运行中的任务，请结束后再启动 H3 Image。\')\n    comfy_process.terminate()\n    comfy_process.wait(timeout=30)\ncomfy_log = open(\'/content/comfy-server.log\', \'a\')\ncomfy_process = subprocess.Popen([sys.executable, \'main.py\', \'--listen\', \'127.0.0.1\', \'--port\', \'8188\', \'--disable-auto-launch\', \'--lowvram\', \'--reserve-vram\', \'2\'], cwd=ROOT, stdout=comfy_log, stderr=subprocess.STDOUT)\nfor attempt in range(90):\n    if comfy_process.poll() is not None:\n        raise RuntimeError(Path(\'/content/comfy-server.log\').read_text()[-10000:])\n    try:\n        info = requests.get(\'http://127.0.0.1:8188/object_info\', timeout=3)\n        if info.ok:\n            assert \'MiniMaxH3ImageConditioning\' in info.json(), \'当前 ComfyUI 缺少 H3 Image 原生节点\'\n            print(\'H3 Image 原生节点已就绪，工作流 MiniMax-H3-Image-L4.json 已保存。\')\n            print(requests.get(\'http://127.0.0.1:8188/system_stats\', timeout=10).json()[\'devices\'])\n            break\n    except requests.RequestException:\n        pass\n    time.sleep(2)\nelse:\n    raise TimeoutError(\'ComfyUI 启动超时，请检查 /content/comfy-server.log\')\n', globals())
print('余额可在 H3 Studio 连接后填写，当前继续启动公网入口。', flush=True)
exec('import os\nimport pathlib\nimport re\nimport secrets\nimport subprocess\nimport time\nimport requests\nfrom IPython.display import display, HTML\nassert requests.get(\'http://127.0.0.1:8188/system_stats\', timeout=10).ok, \'请先运行 H3 Image 服务启动单元\'\nif not pathlib.Path(\'/usr/sbin/nginx\').exists():\n    with open(\'/content/proxy-install.log\', \'w\') as log:\n        subprocess.run([\'apt-get\', \'update\', \'-qq\'], stdout=log, stderr=subprocess.STDOUT, check=True)\n        subprocess.run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'nginx\'], stdout=log, stderr=subprocess.STDOUT, check=True)\ncloudflared = pathlib.Path(\'/content/cloudflared\')\nif not cloudflared.exists():\n    subprocess.run([\'wget\', \'-q\', \'-O\', str(cloudflared), \'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\'], check=True)\n    cloudflared.chmod(493)\nCOMFY_PASSWORD = globals().get(\'COMFY_PASSWORD\') or secrets.token_urlsafe(24)\npassword_hash = subprocess.check_output([\'openssl\', \'passwd\', \'-6\', \'-stdin\'], input=COMFY_PASSWORD + \'\\n\', text=True).strip()\nauth_file = pathlib.Path(\'/content/comfy.htpasswd\')\nauth_file.write_text(\'comfy:\' + password_hash + \'\\n\')\nauth_file.chmod(384)\nconfig_path = pathlib.Path(\'/content/comfy-nginx.conf\')\nif config_path.exists():\n    subprocess.run([\'nginx\', \'-s\', \'quit\', \'-c\', str(config_path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    time.sleep(1)\nconfig_path.write_text(\'user root;\\nworker_processes 1;\\npid /content/comfy-nginx.pid;\\nerror_log /content/comfy-nginx-error.log;\\nevents { worker_connections 1024; }\\nhttp {\\n  access_log off;\\n  map "$request_method:$uri:$http_sec_fetch_mode:$http_sec_fetch_dest" $comfy_fetch_site {\\n    default $http_sec_fetch_site;\\n    "GET:/:navigate:document" none;\\n  }\\n  server {\\n    listen 127.0.0.1:8189;\\n    client_max_body_size 2048m;\\n    auth_basic "ComfyUI";\\n    auth_basic_user_file /content/comfy.htpasswd;\\n    if ($comfy_fetch_site = cross-site) { return 403; }\\n    location / {\\n      proxy_pass http://127.0.0.1:8188;\\n      proxy_http_version 1.1;\\n      proxy_set_header Host $host;\\n      proxy_set_header Sec-Fetch-Site $comfy_fetch_site;\\n      proxy_set_header Upgrade $http_upgrade;\\n      proxy_set_header Connection "upgrade";\\n      proxy_read_timeout 86400;\\n      proxy_send_timeout 86400;\\n      proxy_buffering off;\\n    }\\n  }\\n}\\n\')\nsubprocess.run([\'nginx\', \'-t\', \'-c\', str(config_path)], check=True)\nsubprocess.run([\'nginx\', \'-c\', str(config_path)], check=True)\nassert requests.get(\'http://127.0.0.1:8189/system_stats\', timeout=10).status_code == 401\nassert requests.get(\'http://127.0.0.1:8189/system_stats\', auth=(\'comfy\', COMFY_PASSWORD), timeout=10).ok\nnav_headers = {\'Sec-Fetch-Site\': \'cross-site\', \'Sec-Fetch-Mode\': \'navigate\', \'Sec-Fetch-Dest\': \'document\'}\nassert requests.get(\'http://127.0.0.1:8189/\', headers=nav_headers, timeout=10).status_code == 401\nassert requests.get(\'http://127.0.0.1:8189/\', headers=nav_headers, auth=(\'comfy\', COMFY_PASSWORD), timeout=10).ok\nassert requests.get(\'http://127.0.0.1:8189/system_stats\', headers=nav_headers, auth=(\'comfy\', COMFY_PASSWORD), timeout=10).status_code == 403\nprint(\'H3 Image 本地代理和密码保护已验证。\', flush=True)\n\'Include in the Colab startup cell; requires a Colab secret, never a printed token.\'\nimport hashlib\nimport smtplib\nimport ssl\nfrom email.message import EmailMessage\nCOMFY_NOTIFICATION_EMAIL = H3_NOTIFICATION_EMAIL\n_COMFY_NOTIFIED_LOGINS = globals().get(\'_COMFY_NOTIFIED_LOGINS\', set())\n\ndef get_h3_gmail_password():\n    cached = globals().get(\'H3_GMAIL_APP_PASSWORD\', \'\')\n    if cached:\n        return cached\n    try:\n        from google.colab import userdata\n        value = userdata.get(\'COMFY_GMAIL_APP_PASSWORD\')\n    except Exception as error:\n        reason = type(error).__name__\n        if reason == \'NotebookAccessError\':\n            print(\'已有邮件密钥尚未授权本笔记本，可在左侧 Secret 开启访问权限，或在下方临时填写。\')\n        elif reason == \'SecretNotFoundError\':\n            print(\'当前 Colab 未配置 Gmail 应用专用密码，请通过下方链接生成后填写。\')\n        else:\n            print(\'暂时无法读取 Colab 邮件密钥，可在下方临时填写 Gmail 应用专用密码。\')\n        from IPython.display import display, HTML\n        display(HTML(\'<section><h3>生成 Gmail 应用专用密码</h3><ol><li>登录上方填写的 Gmail 账号，打开 <a href="https://myaccount.google.com/signinoptions/two-step-verification" target="_blank" rel="noopener">Google 两步验证</a>，按页面提示完成开启。</li><li>打开 <a href="https://myaccount.google.com/apppasswords" target="_blank" rel="noopener">应用专用密码</a>。Google 可能要求重新登录。</li><li>在“应用名称”填写 H3 Studio，点击“创建”。</li><li>复制 Google 显示的 16 位应用专用密码，粘贴到下方隐藏输入框并回车。密码中的空格会自动去除。</li></ol><p>这里填写 Google 生成的应用专用密码。临时输入仅用于当前 Colab 运行时，重建运行时后需再次填写。也可将其保存为 Colab Secret COMFY_GMAIL_APP_PASSWORD，并开启本笔记本访问权限。</p></section>\'))\n        import getpass\n        value = getpass.getpass(\'Gmail 应用专用密码（隐藏输入，仅保存在当前运行时内存）：\')\n    value = value.replace(\' \', \'\').strip()\n    if not value:\n        raise RuntimeError(\'未填写 Gmail 应用专用密码。请配置 Secret 或在隐藏输入框填写后重试。\')\n    globals()[\'H3_GMAIL_APP_PASSWORD\'] = value\n    return value\n\ndef send_comfy_login_email(url, username, password):\n    key = hashlib.sha256((url + \'\\n\' + username + \'\\n\' + password).encode()).hexdigest()\n    if key in _COMFY_NOTIFIED_LOGINS:\n        return COMFY_NOTIFICATION_EMAIL\n    app_password = get_h3_gmail_password()\n    message = EmailMessage()\n    message[\'From\'] = COMFY_NOTIFICATION_EMAIL\n    message[\'To\'] = COMFY_NOTIFICATION_EMAIL\n    message[\'Subject\'] = \'ComfyUI 启动登录信息\'\n    message.set_content(f\'ComfyUI 已启动。\\n\\n入口：{url}\\n用户名：{username}\\n密码：{password}\\n\\n此链接随 Colab 运行时或隧道结束而失效。请勿转发本邮件。\')\n    try:\n        with smtplib.SMTP_SSL(\'smtp.gmail.com\', 465, context=ssl.create_default_context(), timeout=30) as smtp:\n            smtp.login(COMFY_NOTIFICATION_EMAIL, app_password)\n            smtp.send_message(message)\n    except (smtplib.SMTPException, OSError):\n        raise RuntimeError(\'邮件发送失败：请检查 Gmail 应用专用密码和网络后重试；密码未写入输出。\') from None\n    _COMFY_NOTIFIED_LOGINS.add(key)\n    return COMFY_NOTIFICATION_EMAIL\nimport pathlib, subprocess, time, re, requests\nfrom IPython.display import display, HTML\nif \'tunnel_process\' not in globals() or tunnel_process.poll() is not None:\n    tunnel_log = open(\'/content/comfy-tunnel.log\', \'w\')\n    tunnel_process = subprocess.Popen([\'/content/cloudflared\', \'tunnel\', \'--url\', \'http://127.0.0.1:8189\', \'--no-autoupdate\', \'--protocol\', \'http2\'], stdout=tunnel_log, stderr=subprocess.STDOUT)\nfor _ in range(60):\n    match = re.search(\'https://[a-z0-9-]+.trycloudflare.com\', pathlib.Path(\'/content/comfy-tunnel.log\').read_text())\n    if match:\n        PUBLIC_URL = match.group(0)\n        break\n    if tunnel_process.poll() is not None:\n        raise RuntimeError(pathlib.Path(\'/content/comfy-tunnel.log\').read_text()[-6000:])\n    time.sleep(1)\nelse:\n    raise TimeoutError(\'未获取到公网链接\')\nfor _ in range(30):\n    try:\n        anon = requests.get(PUBLIC_URL + \'/system_stats\', timeout=15)\n        signed = requests.get(PUBLIC_URL + \'/system_stats\', auth=(\'comfy\', COMFY_PASSWORD), timeout=15)\n        if anon.status_code == 401 and signed.ok:\n            assert signed.json()[\'devices\']\n            print(\'公网连通及密码保护验证通过\')\n            display(HTML(\'<h3><a href="\' + PUBLIC_URL + \'" target="_blank">打开 ComfyUI 公网界面</a></h3>\'))\n            print(\'公网地址：\', PUBLIC_URL)\n            print(\'用户名：comfy\')\n            print(\'密码已发送到：\', send_comfy_login_email(PUBLIC_URL, \'comfy\', COMFY_PASSWORD))\n            print(\'在 ComfyUI 工作流列表打开 MiniMax-H3-Image-L4.json，即可开始生成。\')\n            print(\'结束使用后，请断开并删除 Colab 运行时，停止计算单元消耗。\')\n            break\n    except requests.RequestException:\n        pass\n    time.sleep(2)\nelse:\n    raise RuntimeError(\'隧道已启动，但公网验证暂未通过，请查看 /content/comfy-tunnel.log\')\n', globals())
print('远端已就绪。回到 H3 Studio，使用本次邮件中的地址和密码连接。', flush=True)
